# Tutorial T03 – pandas, librosa (& scikit-learn)
*MIR & ML for Audio · TU Berlin · WiSe 2026/27*

| Block | Topic |
|---|---|
| 0 | Setup |
| 1 | pandas – tables in Python |
| 2 | Open Warngler: Inspect the data before you code |
| 3 | librosa – main feature package in Python |
| 4 | Task: tempo estimation vs. Beatport BPM |
| 5 | scikit-learn – machine learning in Python *(if time allows)* |

In [ ]:
# import standard packages you'll need

import sys
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(sys.executable)

# ─ add your own additional imports here:
import librosa
import librosa.display
from IPython.display import Audio, display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────
CORPUS_DIR = Path("data") / "your-genre-folder"   # ← name of your corpus folder
# ─────────────────────────────────────────────────────────────────────────
SESSION = "T03"
PREVIEW_DIR = CORPUS_DIR / "previews"
OUTPUT_DIR = Path("output") / SESSION      # results of this session
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert CORPUS_DIR.is_dir(), f"Not found: {CORPUS_DIR.resolve()} → check CORPUS_DIR"
CHART_CSV = next(CORPUS_DIR.glob("*.csv"))
print(CHART_CSV)

---
## 1 · pandas – tables in Python

| Excel | pandas |
|---|---|
| Sheet | `DataFrame` (table) |
| Column | `Series` – `df["bpm"]` |
| Row numbers | index – `df.index` |
| Open file | `pd.read_csv("file.csv")` |
| Filter | `df[df["bpm"] > 140]` |
| Sort | `df.sort_values("bpm")` |
| New column with a formula | `df["duration_min"] = df["duration_ms"] / 60_000` |
| COUNTIF / AVERAGE | `(df["bpm"] > 140).sum()`, `df["bpm"].mean()` |
| Pivot table | `df.groupby("year")["bpm"].median()` |
| Count values | `df["label"].value_counts()` |
| VLOOKUP | `df.merge(other, on="id")`, `df["id"].map(lookup)` |
| Save as | `df.to_csv("file.csv", index=False)` |

In [ ]:
df_charts = pd.read_csv(CHART_CSV) # load your csv as df (DataFrame)

print(df_charts.shape)   # (rows, columns)
df_charts.head(10)         # first 5 rows

In [ ]:
df_charts.info()         # columns, dtypes and non-null counts

In [ ]:
# Select columns, filter rows, sort
cols = ["year", "rank", "artist", "title", "bpm"]
top10 = df_charts[df_charts["rank"] <= 10]                  # boolean mask = filter
top10[cols].sort_values("bpm", ascending=False).head(10)

In [ ]:
# New column, summary per group, value counts
df_charts["duration_min"] = (df_charts["duration_ms"] / 60_000).round(2)
print(df_charts.groupby("year")["duration_min"].median())

In [ ]:
# want to save the new column? -> Save as (new) csv, index=False to prevent an index duplicate when reloading the table
df_charts.to_csv(OUTPUT_DIR / "charts_T03.csv", index=False)

**Your turn** – Which are the 5 **longest** tracks in your corpus?

In [ ]:
# your code


---
## 2 look at the data before you code
Open the chart CSV in **Data Wrangler / Open Wrangler** and inspect it before working with scripts:
- What do the columns contain? Which values are missing?
- Sort `bpm` ascending: is the slowest tempo plausible for your genre?

In [ ]:
# Listen to the slowest track 
                        # (filtered by Top 100 only) & only rows with bpm > 0 .nsmallest(number of entries, column)."select first row"
slowest = df_charts[(df_charts["rank"] <= 100) & (df_charts["bpm"] > 0)].nsmallest(1, "bpm") .iloc[0]

print(slowest["artist"], "–", slowest["title"], "|", slowest["bpm"], "BPM")
Audio(filename=PREVIEW_DIR / f"{int(slowest['beatport_track_id'])}.mp3")

---
## 3 · librosa – audio analysis in Python
Key defaults: `librosa.load()` resamples to **22 050 Hz** and downmixes to **mono** (`sr=None`, `mono=False` keep the original). Frames: `n_fft=2048`, `hop_length=512` → one frame every 512 / 22 050 Hz ≈ 23 ms.

Example functions: 
| Module | What it does | Examples |
|---|---|---|
| core / I/O | load, resample, transform | `load`, `stft`, `cqt`, `amplitude_to_db` |
| `feature` | frame-level features | `mfcc`, `spectral_centroid`, `spectral_flatness`, `rms`, `zero_crossing_rate`, `chroma_cqt`, `tempo`, `tempogram` |
| `onset` | onset detection | `onset_strength`, `onset_detect` |
| `segment` | structure analysis | `recurrence_matrix`, `agglomerative` |
| `display` | plots | `waveshow`, `specshow` |

In [ ]:
# Load one preview and look at it: waveform and spectrogram
path = PREVIEW_DIR / f"{int(slowest['beatport_track_id'])}.mp3"
y, sr = librosa.load(path)                                   # mono, 22 050 Hz as default
print(y.shape, sr, f"→ {len(y) / sr:.1f} s")

S_db = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)   # STFT magnitude in dB

# plot waveform and spectrogram
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
librosa.display.waveshow(y, sr=sr, ax=ax1)
img = librosa.display.specshow(S_db, sr=sr, x_axis="time", y_axis="log", ax=ax2)
ax1.set(title="Waveform", ylabel="Amplitude")
ax2.set(title="Spectrogram (log frequency)", xlabel="Time")
fig.colorbar(img, ax=ax2, format="%+2.0f dB")
plt.show()

---
## 4 task: librosa tempo estimation vs. Beatport BPM
`librosa.feature.tempo()` estimates one global tempo per track: onset strength curve → autocorrelation → most likely periodicity, weighted by a prior around `start_bpm`.
Typical error: **octave errors** – half or double tempo (e.g. 87 vs. 174 BPM). Check them on the slowest and fastest tracks of your corpus.

In [ ]:
# 4.1 Select the 10 % slowest Top-100 tracks with a preview (bpm > 0 excludes NaN and the placeholder 0)
tracks = df_charts[(df_charts["rank"] <= 100) & (df_charts["bpm"] > 0)].drop_duplicates("beatport_track_id")
tracks = tracks[[(PREVIEW_DIR / f"{int(i)}.mp3").exists() for i in tracks["beatport_track_id"]]]

# select a percentage of the list
n = round(number=0.10 * len(tracks))   

slow = tracks.nsmallest(n, "bpm")
slow[["artist", "title", "bpm"]].head()

In [ ]:
# 4.2 parameters for tempo estimation
HOP_LENGTH = 512     # samples between frames of the onset curve: 512 @ 22.05 kHz ≈ 23 ms time resolution
AC_SIZE = 8.0        # length of the autocorrelation window in s – should cover several bars
MAX_TEMPO = 200.0    # upper limit of the estimate in BPM
START_BPM = 100.0    # prior: the estimator prefers tempi near this value 

def estimate_tempo(path):
    y, sr = librosa.load(path)
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=HOP_LENGTH)
    return librosa.feature.tempo(onset_envelope=onset_env, sr=sr, hop_length=HOP_LENGTH,
                                 ac_size=AC_SIZE, max_tempo=MAX_TEMPO, start_bpm=START_BPM)[0]

# try it on one track
row = tracks.iloc[0]
print(f"Beatport: {row['bpm']:.0f} BPM | librosa: {estimate_tempo(PREVIEW_DIR / f'{int(row['beatport_track_id'])}.mp3'):.1f} BPM")

In [ ]:
# 4.3 recompute the tempo of the slow tracks with librosa and compare with Beatports bpm
slow["tempo_librosa"] = [estimate_tempo(PREVIEW_DIR / f"{int(i)}.mp3") for i in slow["beatport_track_id"]]
slow["tempo_librosa"] = slow["tempo_librosa"].round(2) 
slow[["artist", "title", "bpm", "tempo_librosa"]]

In [ ]:
# Select a track and listen (use its index for the .loc function)
row = slow.loc[] # insert the selected index here []
print(f"{row['artist']} – {row['title']} | Beatport {row['bpm']:.0f} | librosa {row['tempo_librosa']:.1f}")
Audio(filename=PREVIEW_DIR / f"{int(row['beatport_track_id'])}.mp3")

**Your turn**
- Adjust `HOP_LENGTH`, `AC_SIZE`, `MAX_TEMPO` and `START_BPM` (cell 4.2), rerun 4.3 and compare the results. Which parameter changes the most – and why?
- Listen to some mismatches: is the librosa algorithm wrong, or is the Beatport value off?

In [ ]:
# your code

---
## 5 · scikit-learn – machine learning in Python *(if time allows)*
Every method is an **estimator object** with the same interface:
`model = Method(parameters)` → `model.fit(X)` (learn from data) → `model.transform(X)` / `model.predict(X)` (apply). Shortcuts: `fit_transform`, `fit_predict`.

| Module | Examples | Lecture |
|---|---|---|
| `preprocessing` | `StandardScaler`, `MinMaxScaler`, `OneHotEncoder` | 3, 8 |
| `decomposition` | `PCA`, `NMF` | 3 |
| `cluster`, `mixture` | `KMeans`, `AgglomerativeClustering`, `GaussianMixture` | 3, 6, 7 |
| `model_selection` | `train_test_split`, `cross_val_score`, `GridSearchCV` | 8, 9 |
| `linear_model`, `neighbors`, `ensemble` | `LogisticRegression`, `KNeighborsClassifier`, `RandomForestClassifier` | 10, 11 |
| `metrics` | `accuracy_score`, `r2_score`, `silhouette_score` | 8, 9 |

`X` is always a table of numbers: one row per observation (e.g. track), one column per feature.

In [ ]:
# Example: group tracks by tempo and duration
X = tracks[["bpm", "duration_min"]]                  # features: rows = tracks, columns = features

scaler = StandardScaler()                           # 1) create the estimator
X_z = scaler.fit_transform(X)                       # 2) centering, then transform to z-scores (same scale)

kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)
labels = kmeans.fit_predict(X_z)                    # 3) learn cluster centres, assign each track

tracks.assign(cluster=labels).groupby("cluster")[["bpm", "duration_min"]].median()